# Notebook 6 - Outlier Treatment
## What is an Outlier?

An outlier is a value that sits far away from the rest of the data. That's it.
The word doesn't say why it's far away, and that "why" is the entire job
of this notebook.

## Outlier vs Error

These get confused constantly, but they're not the same thing:

* Error: the value is impossible or a data entry mistake (Age = 999).
* Valid extreme observation: the value is unusual but real (a senior
  employee earning much more than everyone else).

Treating every outlier as an error is how real information gets thrown away.
Treating every outlier as valid is how garbage stays in your dataset.
We'll go column by column and make that call deliberately.

In [1]:
import pandas as pd

df = pd.read_csv("hr_employee_attrition_raw.csv")

df["MonthlyIncome_clean"] = pd.to_numeric(
    df["MonthlyIncome"].astype(str).str.replace(" USD", "", regex=False).str.strip(),
    errors="coerce"
)
df["YearsAtCompany_clean"] = pd.to_numeric(
    df["YearsAtCompany"].astype(str).str.replace(" yrs", "", regex=False).str.strip(),
    errors="coerce"
)

## IQR Method

The IQR (Interquartile Range) method flags anything far outside the middle
50% of the data. It's the most common first pass for outlier detection.

In [3]:
def iqr_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ["Age", "MonthlyIncome_clean", "DistanceFromHomeKM", "YearsAtCompany_clean"]:
    low, high = iqr_bounds(df[col].dropna())
    count = ((df[col] < low) | (df[col] > high)).sum()
    print(f"{col}: bounds=({low:.1f}, {high:.1f}) | outliers={count}")

Age: bounds=(10.0, 58.0) | outliers=16
MonthlyIncome_clean: bounds=(-332.2, 11282.9) | outliers=12
DistanceFromHomeKM: bounds=(-10.8, 23.9) | outliers=72
YearsAtCompany_clean: bounds=(-5.0, 11.0) | outliers=56


## Case 1: Age

IQR flags values above about 58 or below about 10. The worst offenders are
999, 200, and 150.

* Is it an error? Yes. No employee is 999 years old, this is a data
  entry glitch (probably an extra digit typed by mistake).
* Is it a valid extreme observation? No, there's no legitimate reading of
  "age 999" as a real person.
* Should it be removed? No, removing the row loses the rest of that
  employee's data (income, department, tenure) unnecessarily.
* Should it be capped? No, capping would invent a fake age (for example,
  forcing 999 to 58) which is just as wrong as the original error.
* Should it be transformed? No.
* Why: This isn't an outlier problem, it's a data entry error.
  The right fix is correction or treating it as missing, not statistical
  outlier treatment.

In [4]:
# Age errors: treat as missing rather than "outliers", since they're impossible values
df.loc[(df["Age"] < 15) | (df["Age"] > 80), "Age"] = pd.NA
print("Remaining Age nulls to investigate/impute:", df["Age"].isna().sum())

Remaining Age nulls to investigate/impute: 8


## Case 2: DistanceFromHomeKM

IQR flags 72 rows, with a max of 830 km.

* Is it an error? Almost certainly, for the extreme values. An 830 km
  daily commute isn't realistic for an on-site employee.
* Is it a valid extreme observation? Possibly for the moderate high end
  (for example, 40 to 60 km), some employees genuinely live far away.
* Should it be removed? No, same reasoning as Age; don't discard the whole
  row for one bad field.
* Should it be capped? Yes, for the truly extreme values (for example, cap
  at a realistic max like 150 km). This preserves the row while removing the
  impossible part.
* Should it be transformed? Not needed here; capping is enough.
* Why: Most of the 72 flagged rows are probably legitimate long distance
  commuters, and only a handful (like 830 km, 780 km) look like unit errors
  or typos. Blindly removing all 72 would delete a lot of real people.

In [5]:
# Cap unrealistic commute distances instead of deleting the row
cap_value = 150  # realistic upper bound for a daily commute
df["DistanceFromHomeKM_capped"] = df["DistanceFromHomeKM"].clip(upper=cap_value)

print("Rows affected by capping:", (df["DistanceFromHomeKM"] > cap_value).sum())

Rows affected by capping: 6


## Case 3: MonthlyIncome

IQR flags 12 rows, topping out around $71,638 for a "Support Executive" role.

* Is it an error? Possibly, that salary looks unusually high for the
  listed job title.
* Is it a valid extreme observation? Also possible, this could be a
  senior or legacy employee, a data mapping issue between role and pay
  grade, or a real outlier hire.
* Should it be removed? No. Deleting a real employee's salary just
  because it's high would bias any average salary or attrition analysis.
* Should it be capped? Not recommended by default. Capping salary
  distorts pay analysis and could hide a real pay equity issue worth
  investigating.
* Should it be transformed? Yes, for analysis purposes. Apply a log
  transform when modeling, so the high earners don't dominate the scale,
  without altering the original values.
* Why: Income is exactly the kind of column where "extreme" is often
  real, not broken. The safer move is to flag it for review, not silently
  change or delete it.

In [6]:
import numpy as np

# Flag for human review instead of deleting or capping
df["income_flagged_for_review"] = df["MonthlyIncome_clean"] > 11282.9

# Log-transform for modeling/analysis only, original column stays untouched
df["MonthlyIncome_log"] = np.log1p(df["MonthlyIncome_clean"])

print("Rows flagged for manual salary review:", df["income_flagged_for_review"].sum())

Rows flagged for manual salary review: 12


## Case 4: YearsAtCompany

IQR flags 56 rows, with a max of 38 years.

* Is it an error? No, 38 years of tenure is unusual but entirely plausible
  for a long serving employee.
* Is it a valid extreme observation? Yes.
* Should it be removed? No.
* Should it be capped? No, capping would falsify a real, meaningful data
  point (long tenure is often directly relevant to attrition analysis).
* Should it be transformed? Not necessary.
* Why: This is the clearest case of retaining outliers. High tenure
  isn't noise, for an attrition study, it may be one of the most important
  signals in the whole dataset.

In [7]:
# Retain as-is, no treatment needed
df["YearsAtCompany_clean"].describe()

count    1230.000000
mean        3.516260
std         4.026072
min         0.000000
25%         1.000000
50%         2.000000
75%         5.000000
max        38.000000
Name: YearsAtCompany_clean, dtype: float64

## Z-Score Method (Cross-Check)

IQR isn't the only lens. The Z-score method flags values based on standard
deviations from the mean, useful for roughly normal shaped columns like
Age (after fixing the error values) or Income.

In [8]:
from scipy import stats

z_scores = stats.zscore(df["MonthlyIncome_clean"].dropna())
z_outliers = (abs(z_scores) > 3).sum()
print("Z-score outliers (|z| > 3) in MonthlyIncome:", z_outliers)

Z-score outliers (|z| > 3) in MonthlyIncome: 8


## Percentile Method and Winsorization

Instead of statistical bounds, the percentile method simply cuts off the
top or bottom X% of values, useful when you want a fixed, business defined
cutoff rather than a statistically derived one.

Winsorization is a softer alternative to capping: instead of deleting
extreme rows, it replaces them with the nearest acceptable percentile value,
keeping the row but taming the extremity.

In [10]:
from scipy.stats.mstats import winsorize

# Winsorize the top and bottom 1% of income, instead of hard-capping or deleting
df["MonthlyIncome_winsorized"] = winsorize(
    df["MonthlyIncome_clean"].fillna(df["MonthlyIncome_clean"].median()),
    limits=[0.01, 0.01]
)